# 297 nm single-photon CZ comparison

This notebook is a compact report, not an optimizer scratchpad. It compares the fixed first-pass 297 nm CZ operating point against the existing seven-level time-optimal and `find_phase` adiabatic baselines.

Fixed 297 baseline: `P_297 = 0.56 W` at the atoms, `beam_area = 420 um^2`, `B = 100 G`, `nP = 53P3/2`, `spacing = 3 um`. The primary metric is coherent CZ phase/leakage from logical overlaps, plus the trajectory-integrated loss budget (helpers written inline below).

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")
print("BLAS/OpenMP threads pinned to", os.environ.get("OMP_NUM_THREADS"))

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

# Public API only: top-level {Register, RydbergSystem, level_structure, simulate}
# + the CZ-family / Direct297 protocols + forward physics helpers.
import ryd_gate as rg
from ryd_gate import Register, RydbergSystem, level_structure, simulate
from ryd_gate.protocols import (
    CZProtocol, Direct297CZProtocol, Direct297TOProtocol, phase_from_chirp,
)
from ryd_gate.physics import (
    single_photon_rabi, rb87_7_mp_rabi_frequencies, rb87_297_clock_rabi_frequencies,
    zeeman_shift_rad_s, arc_pair_c6_rad_s_um6,
)
from scipy import integrate, interpolate

MHz = 2.0 * np.pi * 1e6
us = 1e-6

SPACING_UM = 3.0
BEAM_AREA_UM2 = 420.0
B_FIELD_G = 100.0
RYD_LEVEL_297 = 53

T_AD = 1.0 * us
D_AMP = 20.0 * MHz
DELTA_CHIRP = 0.0

N_STEPS_AD = 3000   # phase_from_chirp sampling (ODE evolution is adaptive)

RUN_SCANS = True
RUN_297_OPT = False
RUN_DECAY_ON_SANITY = False

if pd is not None:
    pd.options.display.float_format = '{:.6e}'.format


def wrap_to_pi(angle):
    return float((float(angle) + np.pi) % (2.0 * np.pi) - np.pi)



# ---- Inline gate-metric helpers (the formulas live in this notebook) ----

def decay_integrate(t_list, occ_list, decay_rate):
    """Cumulative decay probability P(t) = integral of Gamma*n(tau) dtau.

    Cubic-spline interpolation of the occupation + DOP853 quadrature (the
    convention the cached numbers were produced with; not a trapezoid).
    """
    spline = interpolate.CubicSpline(t_list, occ_list)
    result = integrate.solve_ivp(
        lambda t, _y: np.array([decay_rate * spline(t)]),
        [0, t_list[-1]], np.array([0.0]), t_eval=t_list,
        method="DOP853", rtol=1e-8, atol=1e-12,
    )
    return np.array(result.y)


def _as_labels(initial_state):
    """Accept a label string ("01") or an already-split label list (["0","1"])."""
    return list(initial_state) if isinstance(initial_state, str) else list(initial_state)


def _total_pop(bound, level):
    """ObservableExpr for the total population of ``level`` over all atoms (O08/O11)."""
    obs = bound.observables
    return sum(obs.n(level, i) for i in range(bound.N))


_OCC_LEVEL = {"e1": "e1", "e2": "e2", "e3": "e3", "ryd": "r", "ryd_garb": "r_garb"}


def population_evolution(bound, initial_state, n_eval=1000):
    """Per-level population time series on a bound 7-level system (E09 dense n_k(t))."""
    obs = {key: _total_pop(bound, lvl) for key, lvl in _OCC_LEVEL.items()}
    res = simulate(bound, _as_labels(initial_state),
                   t_eval=np.linspace(0.0, bound.t_gate, n_eval), observables=obs)
    out = {"t_list": np.asarray(res.times)}
    for key in _OCC_LEVEL:
        out[key] = np.asarray(res.expectation(key))
    return out


def population_evolution_297(bound, initial_state, n_eval=1000):
    """Rydberg population time series on a bound 297 nm 4-level system."""
    obs = {"ryd": _total_pop(bound, "r"), "ryd_garb": _total_pop(bound, "r_garb")}
    res = simulate(bound, _as_labels(initial_state),
                   t_eval=np.linspace(0.0, bound.t_gate, n_eval), observables=obs)
    return {
        "t_list": np.asarray(res.times),
        "ryd": np.asarray(res.expectation("ryd")),
        "ryd_garb": np.asarray(res.expectation("ryd_garb")),
    }


def error_budget(bound, initial_states=("01", "11")):
    """7-level XYZ/AL/LG budget by decay source, averaged over initial states (A01/A02).

    Coherent-only model: run each trajectory and integrate Gamma_k*<n_k(t)>.
    Radiative Rydberg decay splits into the qubit (XYZ) vs the other hyperfine
    ground states (LG) by the ARC branching ratios; BBR + final residual Rydberg
    population count as atom loss (AL). Intermediate-manifold decay and residuals
    split by the per-F 6P branching. Rates on decay_rates_per_s, branching on
    branching_ratios.
    """
    ls = bound.level_structure
    ryd_rd = ls.decay_rates_per_s["r"]["radiative"]
    ryd_bbr = ls.decay_rates_per_s["r"]["blackbody"]
    mid_total = ls.decay_rates_per_s["e1"]["total"]
    br = ls.branching_ratios["r"]
    xyz_frac, lg_frac = br["to_0"] + br["to_1"], br["to_L0"] + br["to_L1"]
    budget = {src: {"XYZ": 0.0, "AL": 0.0, "LG": 0.0}
              for src in ("rydberg_decay", "intermediate_decay", "polarization_leakage")}
    for s in initial_states:
        pops = population_evolution(bound, s)
        t = pops["t_list"]
        for src, occ in (("rydberg_decay", pops["ryd"]),
                         ("polarization_leakage", pops["ryd_garb"])):
            rd = decay_integrate(t, occ, ryd_rd)[0, -1]
            bbr = decay_integrate(t, occ, ryd_bbr)[0, -1]
            budget[src]["XYZ"] += rd * xyz_frac
            budget[src]["LG"] += rd * lg_frac
            budget[src]["AL"] += bbr + occ[-1]
        for lvl in ("e1", "e2", "e3"):
            occ = pops[lvl]
            mid_tot = decay_integrate(t, occ, mid_total)[0, -1] + occ[-1]
            mbr = ls.branching_ratios[lvl]
            budget["intermediate_decay"]["XYZ"] += mid_tot * (mbr["to_0"] + mbr["to_1"])
            budget["intermediate_decay"]["LG"] += mid_tot * (mbr["to_L0"] + mbr["to_L1"])
    n = len(initial_states)
    return {src: {"total": sum(vals.values()) / n,
                  **{k: v / n for k, v in vals.items()}}
            for src, vals in budget.items()}


def error_budget_297(bound, initial_states=("01", "10", "11")):
    """Flat 297 decay/residual budget: Gamma*integral(n dt) + final residuals."""
    gamma = float(bound.level_structure.decay_rates_per_s["r"]["total"])
    accum = dict.fromkeys(("p_ryd_decay", "p_target_ryd_decay", "p_garb_decay",
                           "p_ryd_residual", "p_garb_residual"), 0.0)
    for s in initial_states:
        pops = population_evolution_297(bound, s)
        t = pops["t_list"]
        p_target = decay_integrate(t, pops["ryd"], gamma)[0, -1]
        p_garb = decay_integrate(t, pops["ryd_garb"], gamma)[0, -1]
        accum["p_target_ryd_decay"] += p_target
        accum["p_garb_decay"] += p_garb
        accum["p_ryd_decay"] += p_target + p_garb
        accum["p_ryd_residual"] += pops["ryd"][-1]
        accum["p_garb_residual"] += pops["ryd_garb"][-1]
    n = len(initial_states)
    out = {k: float(v / n) for k, v in accum.items()}
    out["p_total"] = out["p_ryd_decay"] + out["p_ryd_residual"] + out["p_garb_residual"]
    return out


In [ ]:
OPTICS_LOSS = 0.8       # 80% loss of power
Delta = 40.1e3 * MHz  # rad/s, the intermediate detuning used to build sys7

beam_area_um2 = 7 * 20 * SPACING_UM
omega_420, omega_1013 = rb87_7_mp_rabi_frequencies(
    6 * (1 - OPTICS_LOSS),            # 420 nm 功率 (W)
    100 * (1 - OPTICS_LOSS),          # 1013 nm 功率 (W)
    beam_area_um2,                    # 光斑面积 (μm²)
    ryd_level=70,
)
print(f"omega_420/2pi = {omega_420/MHz:.3f} MHz, omega_1013/2pi = {omega_1013/MHz:.3f} MHz")

# Verification
res = single_photon_rabi(
        power_w =100.0*(1 - OPTICS_LOSS), beam_area_um2 = beam_area_um2,
        n1=6, l1=1, j1=1.5, mj1=-0.5, n2=70, l2=0, j2=0.5, q=1,
    )/ single_photon_rabi(
        power_w =100.0*(1 - OPTICS_LOSS), beam_area_um2 = beam_area_um2,
        n1=6, l1=1, j1=1.5, mj1=-1.5, n2=70, l2=0, j2=0.5, q=1,
    )

print(res**2)

p297_w = 2.8*(1 - OPTICS_LOSS)
# Comparision: single qubit laser: 297
prop_plus = single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=-0.5, n2=53, l2=1, j2=1.5, q=-1,
    )/single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=0.5, n2=53, l2=1, j2=1.5, q=-1,
    )
prop_minus = single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=-0.5, n2=53, l2=1, j2=1.5, q=1,
    )/single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=0.5, n2=53, l2=1, j2=1.5, q=1,
    )
prop_pi = single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=-0.5, n2=53, l2=1, j2=1.5, q=0,
    )/single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=0.5, n2=53, l2=1, j2=1.5, q=0,
    )

prop_297 = single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=-0.5, n2=53, l2=1, j2=1.5, q=-1,
    )/single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=0.5, n2=53, l2=1, j2=0.5, q=-1,
    )

print(prop_plus**2,prop_minus**2,prop_pi**2,prop_297**2)

# explicit power
omega_297 = single_photon_rabi(
        power_w = p297_w, beam_area_um2 = beam_area_um2,
        n1=5, l1=0, j1=0.5, mj1=-0.5, n2=53, l2=1, j2=1.5, q=-1,
    )/2
print(f"omega_297/2pi = {omega_297/MHz:.3f} MHz, omega_eff/2pi = {omega_1013*omega_420/(4*Delta*MHz):.3f} MHz")


In [ ]:
#   g_J = 1 + (J(J+1) + S(S+1) - L(L+1)) / (2 J(J+1))
#   53P3/2: g_J = 4/3  -> Delta/2pi ≈14.929 MHz
#   70S1/2: g_J ≈ 2    -> Delta/2pi ≈ 22.394 MHz

#   所以它和 n=53 或 n=70 无关，但和 S1/2 vs P3/2 有关。
B = 8.0
singpho_detuning = zeeman_shift_rad_s(
      B, l=1, j=1.5,
      delta_mj=1.0
  )
doubpho_detuning = zeeman_shift_rad_s(
      B, l=0, j=0.5,
      delta_mj=1.0
  )
print(f"single_photon_detuning/2pi = {singpho_detuning/MHz:.3f} MHz")
print(f"double_photon_detuning/2pi = {doubpho_detuning/MHz:.3f} MHz")

## Pulse builders

The 297 adiabatic pulse uses a bare phase chirp. The seven-level adiabatic baseline follows `find_phase.ipynb`: smooth 420/1013 envelopes and the Stark-compensated 420 chirp.

In [ ]:
OPTICS_LOSS = 0.8

def env(t, ramp=0.15):
    s = float(np.clip(t / T_AD, 0.0, 1.0))
    u = min(s, 1.0 - s)
    if u >= ramp:
        return 1.0
    u = float(np.clip(u / ramp, 0.0, 1.0))
    return 10.0 * u**3 - 15.0 * u**4 + 6.0 * u**5

chirp297 = lambda t: -D_AMP * np.cos(2.0 * np.pi * t / T_AD) + DELTA_CHIRP
phi297 = phase_from_chirp(chirp297, T_AD, n_samples=4 * N_STEPS_AD + 1)

P297_W = 3

# Power -> target |1>-|r> 297 nm Rabi (rad/s); the garbage-branch coupling comes
# from the preset's leg ratio, so only omega_297_max_rad_s is passed to the protocol.
omega297, omega297_garb = rb87_297_clock_rabi_frequencies(
    P297_W * (1.0 - OPTICS_LOSS), BEAM_AREA_UM2, ryd_level=RYD_LEVEL_297
)

# Direct297CZProtocol takes an explicit omega_297_max_rad_s + physical-time
# envelope/phase callables (P28); clip to [0, T_AD] so a boundary float never
# leaves the phase_from_chirp domain.
proto297 = Direct297CZProtocol(
    t_gate_s=T_AD,
    omega_297_max_rad_s=omega297,
    envelope_297=lambda t: env(float(np.clip(t, 0.0, T_AD))),
    phase_297_rad=lambda t: phi297(float(np.clip(t, 0.0, T_AD))),
)

DELTA_7 = 40.1e3 * MHz
omega420, omega1013 = rb87_7_mp_rabi_frequencies(
    6.0 * (1.0 - OPTICS_LOSS),
    100.0 * (1.0 - OPTICS_LOSS),
    BEAM_AREA_UM2,
    ryd_level=70,
)

def A420_t(t):
    return env(float(np.clip(t, 0.0, T_AD)), ramp=0.15)

def A1013_t(t):
    return env(float(np.clip(t, 0.0, T_AD)), ramp=0.05)

def chirp7_with_stark(t):
    a = A420_t(t)
    b = A1013_t(t)
    d1_nom = -(4.0 / 3.0) * omega420**2 / (4.0 * DELTA_7)
    dr_nom = -(omega1013**2) / (4.0 * DELTA_7)
    return -D_AMP * np.cos(2.0 * np.pi * t / T_AD) + dr_nom * b * b - d1_nom * a * a + DELTA_CHIRP


phi7 = phase_from_chirp(chirp7_with_stark, T_AD, n_samples=4 * N_STEPS_AD + 1)

# CZProtocol: explicit t_gate_s + intermediate_detuning_rad_s + peak Rabis, plus
# physical-time 420/1013 envelope + phase callables (P19/P20/P24). The 420 phase
# is the Stark-compensated chirp integral; the 1013 phase is flat.
proto = CZProtocol(
    t_gate_s=T_AD,
    intermediate_detuning_rad_s=DELTA_7,
    omega_420_max_rad_s=omega420,
    omega_1013_max_rad_s=omega1013,
    envelope_420=lambda t: A420_t(float(np.clip(t, 0.0, T_AD))),
    phase_420_rad=lambda t: phi7(float(np.clip(t, 0.0, T_AD))),
    envelope_1013=lambda t: A1013_t(float(np.clip(t, 0.0, T_AD))),
    phase_1013_rad=lambda t: 0.0,
)

print(f"omega_297/2pi = {omega297/MHz:.3f} MHz, omega_garb/2pi = {omega297_garb/MHz:.3f} MHz, "
      f"omega_420/2pi = {omega420/MHz:.3f} MHz, omega_1013/2pi = {omega1013/MHz:.3f} MHz, "
      f"omegaeff/2pi = {omega420*omega1013/(4*DELTA_7*MHz):.3f} MHz")


In [ ]:
geom = Register.chain(2, spacing_um=SPACING_UM)

# The intermediate detuning is now a protocol parameter (baked into proto above),
# so the preset is purely atomic: only the magnetic field (and, for 297, the
# Rydberg level + quantization axis) are physical kwargs.
level7 = level_structure("rb87_7_mp", magnetic_field_G=B)
level297 = level_structure("rb87_297_clock_4", magnetic_field_G=B, ryd_level=RYD_LEVEL_297)

sys7 = RydbergSystem(level_structure=level7, register=geom, protocol=proto)
sys297 = RydbergSystem(level_structure=level297, register=geom, protocol=proto297)

# Nearest-neighbour pair strengths V = C6/R^6 (the interaction is intrinsic to the
# preset now; there is no interaction_pairs accessor). 7L is the isotropic 70S1/2
# S-state C6; 297 is the channel-resolved 53P3/2 |mj=-3/2> pair at the chain
# geometry (chain along x, quantization axis z -> theta = pi/2).
dim7 = len(level7.levels) ** sys7.N
dim297 = len(level297.levels) ** sys297.N
V7 = arc_pair_c6_rad_s_um6(n1=70, l1=0, j1=0.5, mj1=-0.5, mj2=-0.5,
                           theta=0.0, phi=0.0, degenerate=False) / SPACING_UM**6
V297 = arc_pair_c6_rad_s_um6(n1=RYD_LEVEL_297, l1=1, j1=1.5, mj1=-1.5, mj2=-1.5,
                             theta=np.pi / 2, phi=0.0) / SPACING_UM**6
print(f"7L: dim={dim7}, V_nn/2pi={V7 / MHz:.3f} MHz")
print(f"297: dim={dim297}, V_nn/2pi={V297 / MHz:.3f} MHz")

# Inline pulse-schedule plots (protocols carry no .plot; S14). Amplitudes are the
# dimensionless envelopes; the phases are the (Stark-compensated / bare) chirps.
tt = np.linspace(0.0, T_AD, 400)
fig, axs = plt.subplots(1, 2, figsize=(11.0, 3.4))
axs[0].plot(tt / us, [A420_t(t) for t in tt], label="420 envelope")
axs[0].plot(tt / us, [A1013_t(t) for t in tt], label="1013 envelope")
axs[0].plot(tt / us, [chirp7_with_stark(t) / MHz for t in tt], label="420 chirp /2pi (MHz)")
axs[0].set_title("7-level CZ schedule")
axs[1].plot(tt / us, [env(t) for t in tt], label="297 envelope")
axs[1].plot(tt / us, [chirp297(t) / MHz for t in tt], label="297 chirp /2pi (MHz)")
axs[1].set_title("297 nm CZ schedule")
for ax in axs:
    ax.set_xlabel("time (us)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


In [ ]:
labels = ["00", "01", "10", "11"]
cfgs = [list(s) for s in labels]
N_EVAL_POP = 301


def simulate_logical_basis(system, n_eval=N_EVAL_POP):
    t_eval = np.linspace(0.0, system.t_gate, n_eval)
    observables = {"sum_n_r": _total_pop(system, "r"),
                   "sum_n_r_garb": _total_pop(system, "r_garb")}
    return simulate(system, cfgs, t_eval=t_eval, observables=observables)


def logical_basis_report(system, results, title):
    """CZ overlaps / leakage / phases read off result.amplitude (W01/W02)."""
    phases = {}
    print(title)
    print(" s  | return prob | leakage  | phi_full")
    for j, s in enumerate(labels):
        r = results[j]
        ov = r.amplitude(cfgs[j])                                   # <s|U|s>
        logical_prob = sum(abs(r.amplitude(list(o))) ** 2 for o in labels)
        leak = 1.0 - logical_prob
        phases[s] = float(np.angle(ov))
        print(f" {s} |  {abs(ov)**2:.6f}  | {leak:.2e} | {phases[s]:+.5f}")
    zz = wrap_to_pi(phases["11"] - phases["01"] - phases["10"] + phases["00"])
    print("ZZ phase: ", zz)
    return phases, zz


def plot_metric_populations(populations, level_specs, *, n_atoms, title):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    for ax, s in zip(axes.ravel(), labels):
        pops = populations[s]
        time_us = np.asarray(pops["t_list"], dtype=float) / us
        for name, key, color in level_specs:
            if isinstance(key, tuple):
                y = sum(np.asarray(pops[k], dtype=float) for k in key)
            else:
                y = np.asarray(pops[key], dtype=float)
            ax.plot(time_us, y / n_atoms, lw=1.2, color=color, label=name)
        ax.set_title(f"initial $|{s}\\rangle$")
        ax.set_ylim(-0.01, 1.05)
        ax.set_xlabel("time (us)")
        ax.set_ylabel("population / atom")
        ax.legend(fontsize=7, loc="upper right")
    fig.suptitle(title, y=0.995)
    fig.tight_layout()
    plt.show()


# Runtime note: each 7-level exact_ode solve at Delta_e ~ 40 GHz takes several
# minutes at default tolerances; the 4 basis states + 4 population traces below
# add up to tens of minutes.
results7 = simulate_logical_basis(sys7)
phi7_full, zz7 = logical_basis_report(sys7, results7, "7-level")

pop7 = {s: population_evolution(sys7, s) for s in labels}
level_specs7 = [
    ("intermediate", ("e1", "e2", "e3"), "tab:blue"),
    (r"Rydberg $|r\rangle$", "ryd", "tab:red"),
    (r"garbage $|r_2\rangle$", "ryd_garb", "gray"),
]
plot_metric_populations(
    pop7,
    level_specs7,
    n_atoms=sys7.N,
    title="7-level excited-state population evolution (inline population_evolution)",
)


In [ ]:
results297 = simulate_logical_basis(sys297)
phi297_full, zz297 = logical_basis_report(sys297, results297, "297 nm 4-level")

pop297 = {s: population_evolution_297(sys297, s) for s in labels}
level_specs297 = [
    (r"Rydberg $|r\rangle$", "ryd", "tab:red"),
    (r"garbage $|r_2\rangle$", "ryd_garb", "gray"),
]
plot_metric_populations(
    pop297,
    level_specs297,
    n_atoms=sys297.N,
    title="297 nm Rydberg population evolution (inline population_evolution_297)",
)


In [ ]:
# Integrated loss estimates from the trajectory populations (inline helpers).
# These are first-order event probabilities, not a non-Hermitian norm loss.

Gamma_e = float(sys7.level_structure.decay_rates_per_s["e1"]["total"])
Gamma_r = float(sys7.level_structure.decay_rates_per_s["r"]["total"])
Gamma_r_garb = float(sys7.level_structure.decay_rates_per_s["r_garb"]["total"])

loss_curves7 = {}

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, s in zip(axes.ravel(), labels):
    pops = pop7[s]
    t = np.asarray(pops["t_list"], dtype=float)
    ne = np.asarray(pops["e1"] + pops["e2"] + pops["e3"], dtype=float)
    nr = np.asarray(pops["ryd"], dtype=float)
    nrg = np.asarray(pops["ryd_garb"], dtype=float)

    p_mid = decay_integrate(t, ne, Gamma_e)[0]
    p_ryd = decay_integrate(t, nr, Gamma_r)[0]
    p_r_garb = decay_integrate(t, nrg, Gamma_r_garb)[0]
    p_total = p_mid + p_ryd + p_r_garb
    loss_curves7[s] = {
        "mid": p_mid,
        "ryd": p_ryd,
        "r_garb": p_r_garb,
        "total": p_total,
        "ne": ne,
        "nr": nr,
        "nrg": nrg,
    }

    ax.plot(t / us, p_mid, lw=1.4, color="tab:blue", label="mid-state scattering")
    ax.plot(t / us, p_ryd, lw=1.4, color="tab:red", label=r"Rydberg $|r\rangle$ decay")
    ax.plot(t / us, p_r_garb, lw=1.4, color="gray", label=r"garbage $|r_2\rangle$ decay")
    ax.plot(t / us, p_total, lw=1.8, color="black", ls="--", label="total")
    ax.set_title(f"initial $|{s}\\rangle$")
    ax.set_xlabel("time (us)")
    ax.set_ylabel("cumulative loss probability")
    ax.set_ylim(bottom=0.0)
    ax.legend(fontsize=7, loc="upper left")

fig.suptitle("Integrated mid-state and Rydberg loss estimates", y=0.995)
fig.tight_layout()
plt.show()

print(f"Gamma_e = {Gamma_e:.6e} s^-1")
print(f"Gamma_r = {Gamma_r:.6e} s^-1")
print(" s  | p_mid      | p_ryd      | p_r_garb   | p_total")
for s in labels:
    p_mid = loss_curves7[s]["mid"][-1]
    p_ryd = loss_curves7[s]["ryd"][-1]
    p_r_garb = loss_curves7[s]["r_garb"][-1]
    p_total = loss_curves7[s]["total"][-1]
    print(f" {s} | {p_mid:.3e} | {p_ryd:.3e} | {p_r_garb:.3e} | {p_total:.3e}")

budget7 = error_budget(sys7, initial_states=["01", "11"])
print("\nerror_budget average over |01>, |11>")
print(" source                 | total     | XYZ       | AL        | LG")
for source, row in budget7.items():
    print(
        f" {source:<22} | {row['total']:.3e} | {row['XYZ']:.3e} | "
        f"{row['AL']:.3e} | {row['LG']:.3e}"
    )


In [ ]:
# Integrated loss estimates from the 297 nm four-level trajectories.
# 297 has no intermediate manifold; both |r> and |r_garb> use the nP decay rate.
# The plotted curves are cumulative decay probabilities. The printed p_total
# additionally includes final residual Rydberg population, matching error_budget_297.

Gamma_297 = float(sys297.level_structure.decay_rates_per_s["r"]["total"])

loss_curves297 = {}

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, s in zip(axes.ravel(), labels):
    pops = pop297[s]
    t = np.asarray(pops["t_list"], dtype=float)
    nr = np.asarray(pops["ryd"], dtype=float)
    nrg = np.asarray(pops["ryd_garb"], dtype=float)

    p_target = decay_integrate(t, nr, Gamma_297)[0]
    p_garb = decay_integrate(t, nrg, Gamma_297)[0]
    p_ryd_decay = p_target + p_garb
    p_ryd_residual = float(nr[-1])
    p_garb_residual = float(nrg[-1])
    p_total = float(p_ryd_decay[-1] + p_ryd_residual + p_garb_residual)
    loss_curves297[s] = {
        "target_ryd_decay": p_target,
        "garb_decay": p_garb,
        "ryd_decay": p_ryd_decay,
        "p_ryd_residual": p_ryd_residual,
        "p_garb_residual": p_garb_residual,
        "p_total": p_total,
        "nr": nr,
        "nrg": nrg,
    }

    ax.plot(t / us, p_target, lw=1.4, color="tab:red", label=r"target $|r\rangle$ decay")
    ax.plot(t / us, p_garb, lw=1.4, color="gray", label=r"garbage $|r_2\rangle$ decay")
    ax.plot(t / us, p_ryd_decay, lw=1.8, color="black", ls="--", label="decay total")
    ax.set_title(f"initial $|{s}\\rangle$")
    ax.set_xlabel("time (us)")
    ax.set_ylabel("cumulative decay probability")
    ax.set_ylim(bottom=0.0)
    ax.legend(fontsize=7, loc="upper left")

fig.suptitle("Integrated 297 nm Rydberg loss estimates", y=0.995)
fig.tight_layout()
plt.show()

print(f"Gamma_297 = {Gamma_297:.6e} s^-1")
print(" s  | p_target   | p_garb     | p_ryd_res  | p_garb_res | p_total")
for s in labels:
    p_target = loss_curves297[s]["target_ryd_decay"][-1]
    p_garb = loss_curves297[s]["garb_decay"][-1]
    p_ryd_residual = loss_curves297[s]["p_ryd_residual"]
    p_garb_residual = loss_curves297[s]["p_garb_residual"]
    p_total = loss_curves297[s]["p_total"]
    print(
        f" {s} | {p_target:.3e} | {p_garb:.3e} | "
        f"{p_ryd_residual:.3e} | {p_garb_residual:.3e} | {p_total:.3e}"
    )

budget297 = error_budget_297(sys297, initial_states=["01", "10", "11"])
print("\nerror_budget_297 average over |01>, |10>, |11>")
for key, value in budget297.items():
    print(f" {key:<20} = {value:.3e}")
